# Baseline HTR sur CATMuS Medieval — Old/Middle French XIIIᵉ-XVᵉ

**Projet** : Volet 1 — Transcription automatique de manuscrits médiévaux français
**Équipe** : MD5 HETIC 2026
**Auteur** : `<à remplir>`
**Date** : `<à remplir>`

## Objectif de ce notebook

1. Charger le dataset **CATMuS Medieval** depuis HuggingFace
2. **Filtrer** le sous-corpus de travail (Old/Middle French, XIIIᵉ-XVᵉ siècle)
3. **Sceller** le test set et calculer son hash **SHA-256**
4. Évaluer la **baseline TrOCR-base-handwritten en zéro-shot** sur un échantillon
5. Logger les résultats dans `experiments/journal.jsonl`

## Comment l'utiliser

Faire tourner toutes les cellules dans l'ordre (`Runtime → Run all`). Sur Colab gratuit (T4 16GB), prévoir ~15 minutes. À la fin vous aurez un CER baseline défendable et un hash de test set scellé.

## Avertissement

La baseline TrOCR-base-handwritten est entraînée sur de l'écriture moderne. Sur du français médiéval, **le CER attendu est très mauvais (probablement > 80 %)**. C'est normal, c'est précisément ce qui justifie le fine-tuning.


## 1. Installation des dépendances

Si vous êtes en local avec un environnement déjà préparé, vous pouvez skipper cette cellule.

In [ ]:
# Installation (~3 min sur Colab)
%pip install -q transformers==4.40.1 datasets==2.19.0 peft==0.10.0 \
    torch==2.2.2 torchvision==0.17.2 \
    accelerate==0.29.3 huggingface_hub==0.23.0 \
    jiwer==3.0.4 editdistance==0.8.1 pillow==10.3.0
print("✓ Dépendances installées")

## 2. Imports et fixation des seeds

**Règle d'or** : `fixer_seeds(42)` en début de chaque script ou notebook.

In [ ]:
import hashlib
import json
import random
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm

# Fixer les seeds pour la reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Device : {DEVICE}")
print(f"✓ Seed   : {SEED}")
if DEVICE == "cuda":
    print(f"✓ GPU    : {torch.cuda.get_device_name(0)}")

## 3. Chargement de CATMuS Medieval

Le dataset complet fait environ 2 Go. Premier téléchargement ~5 minutes selon la connexion.

> **Note** : le dataset contient une colonne `gen_split` qui définit le split natif train/val/test par manuscrit (90/5/5). On l'utilise pour éviter toute fuite scripteur.

In [ ]:
print("Téléchargement de CATMuS Medieval (premier load = quelques minutes)...")
ds = load_dataset("CATMuS/medieval")
print()
print("✓ Dataset chargé")
print(f"Splits disponibles : {list(ds.keys())}")
print()
# Inspecter la structure
main_split = list(ds.keys())[0]
print(f"Colonnes du split '{main_split}' :")
for col in ds[main_split].column_names:
    print(f"  - {col}")
print()
print(f"Nombre total d'exemples : {len(ds[main_split])}")
print()
print("Aperçu du premier exemple :")
example = ds[main_split][0]
for k, v in example.items():
    if k == "im":
        print(f"  {k}: <PIL Image {v.size if hasattr(v, 'size') else type(v)}>")
    elif isinstance(v, str) and len(v) > 80:
        print(f"  {k}: {v[:80]}...")
    else:
        print(f"  {k}: {v}")

## 4. Exploration de la distribution

On regarde la répartition par langue et par siècle pour choisir notre filtre.

In [ ]:
# Récupère le DataFrame des métadonnées (sans les images pour aller vite)
import pandas as pd

# Tente plusieurs noms de colonnes possibles selon la version du dataset
df = ds[main_split].remove_columns([c for c in ["im"] if c in ds[main_split].column_names]).to_pandas()
print(f"Colonnes méta : {df.columns.tolist()}")
print()

# Trouver la colonne langue (selon version : 'language' ou 'lang')
lang_col = next((c for c in ["language", "lang"] if c in df.columns), None)
# Trouver la colonne siècle
century_col = next((c for c in ["century", "siecle"] if c in df.columns), None)
# Trouver la colonne split
split_col = next((c for c in ["gen_split", "split"] if c in df.columns), None)

print(f"Colonne langue   : {lang_col}")
print(f"Colonne siècle   : {century_col}")
print(f"Colonne split    : {split_col}")
print()

if lang_col:
    print("Distribution par langue :")
    print(df[lang_col].value_counts().head(15))
    print()
if century_col:
    print("Distribution par siècle :")
    print(df[century_col].value_counts().sort_index())
    print()
if split_col:
    print("Distribution par split natif :")
    print(df[split_col].value_counts())

## 5. Filtrage du sous-corpus

**Filtre retenu** : Old French + Middle French (codes ISO `fro` et `frm`), siècles XIIIᵉ-XVᵉ.

> ⚠️ Si les noms de codes / colonnes diffèrent dans votre version du dataset, ajustez les variables ci-dessous. La cellule précédente vous montre les valeurs disponibles.

In [ ]:
# CONFIG À AJUSTER selon ce que la cellule précédente a affiché
TARGET_LANGUAGES = ["fro", "frm", "Old French", "Middle French", "fre"]  # variantes possibles
TARGET_CENTURIES = [13, 14, 15]


def filter_subcorpus(df, lang_col, century_col):
    """Filtre le DataFrame sur les critères du sous-corpus."""
    mask = pd.Series([True] * len(df))
    if lang_col:
        mask &= df[lang_col].astype(str).isin([str(x) for x in TARGET_LANGUAGES])
    if century_col:
        # Tolère les valeurs numériques ou strings type "Century:14"
        century_int = df[century_col].astype(str).str.extract(r"(\d+)").astype(float)[0]
        mask &= century_int.isin(TARGET_CENTURIES)
    return df[mask].copy()


sub_df = filter_subcorpus(df, lang_col, century_col)
print(f"Sous-corpus filtré : {len(sub_df)} lignes (sur {len(df)} initiales)")
print(f"Soit {100 * len(sub_df) / len(df):.1f} % du corpus total")
print()

if len(sub_df) == 0:
    print("⚠️  AUCUNE LIGNE après filtrage. Vérifiez les valeurs des colonnes ci-dessus.")
    print("    Adaptez TARGET_LANGUAGES et TARGET_CENTURIES en conséquence.")
elif lang_col and century_col and split_col:
    print("Répartition du sous-corpus par split / siècle :")
    print(pd.crosstab(sub_df[century_col].astype(str), sub_df[split_col], margins=True))

## 6. Scellage du test set — SHA-256

⚠️ **Cette opération est définitive.** Après cette cellule, le test set ne doit plus jamais être inspecté avant le rendu final.

Le hash SHA-256 est la garantie de non-contamination demandée par le brief.

In [ ]:
# Construire le test set à partir du split natif
if split_col and len(sub_df) > 0:
    test_indices = sub_df[sub_df[split_col].astype(str).str.lower().isin(["test"])].index.tolist()
    val_indices = sub_df[sub_df[split_col].astype(str).str.lower().isin(["validation", "val", "dev"])].index.tolist()
    train_indices = sub_df[sub_df[split_col].astype(str).str.lower().isin(["train"])].index.tolist()

    print(f"Train : {len(train_indices):>6} lignes")
    print(f"Val   : {len(val_indices):>6} lignes")
    print(f"Test  : {len(test_indices):>6} lignes")
    print()

    # Calcul du SHA-256 du test set
    # On hash sur les contenus textuels triés (pour stabilité, indépendant de l'ordre des indices)
    test_texts = sorted([str(df.loc[i, "text"]) for i in test_indices if "text" in df.columns])
    test_signature = "\n".join(test_texts).encode("utf-8")
    test_hash = hashlib.sha256(test_signature).hexdigest()

    print(f"SHA-256 du test set : {test_hash}")
    print()
    print("📋 COPIE CE HASH dans DATA_SOURCES.md et MODEL_CARD.md")
    print("   À partir de maintenant, le test set est SCELLÉ.")
else:
    print("⚠️  Impossible de constituer les splits. Vérifiez la cellule précédente.")
    test_hash = None

## 7. Charger TrOCR-base-handwritten

C'est notre modèle de base. On l'évalue zéro-shot pour avoir un CER de départ.

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

print("Chargement de TrOCR-base-handwritten (~1.3 GB la première fois)...")
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")
model = model.to(DEVICE)
model.eval()
print(f"✓ Modèle chargé sur {DEVICE}")
print(f"  Paramètres totaux : {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")

## 8. Inférence baseline sur un échantillon de validation

On échantillonne 100 lignes du **val** set (pas du test scellé !). Sur Colab T4, ça prend ~3-5 minutes.

In [ ]:
SAMPLE_SIZE = 100  # à augmenter à 500-1000 pour un chiffre plus stable

# Reconstruire un dataset depuis les indices val (avec les images)
val_dataset = ds[main_split].select(val_indices)
sample_size = min(SAMPLE_SIZE, len(val_dataset))
sample = val_dataset.shuffle(seed=SEED).select(range(sample_size))

print(f"Inférence sur {sample_size} lignes de validation...")

predictions = []
references = []

start = time.time()
for example in tqdm(sample, desc="TrOCR zéro-shot"):
    image = example["im"]
    if image.mode != "RGB":
        image = image.convert("RGB")
    reference = example["text"]

    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(DEVICE)
    with torch.no_grad():
        generated_ids = model.generate(pixel_values, max_length=128)
    prediction = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    predictions.append(prediction)
    references.append(reference)

elapsed = time.time() - start
print(f"✓ Terminé en {elapsed:.1f}s ({elapsed/sample_size*1000:.0f} ms par ligne)")

## 9. Calcul du CER et du WER

Le CER (Character Error Rate) est notre métrique principale. C'est la distance de Levenshtein normalisée par la longueur de la référence.

In [ ]:
import editdistance


def cer(predictions, references):
    """Character Error Rate global (distance Levenshtein / longueur référence).

    Args:
        predictions: Liste des transcriptions prédites.
        references: Liste des transcriptions de référence.

    Returns:
        float: CER global entre 0 et 1+ (peut dépasser 1 si beaucoup d'insertions).
    """
    total_distance = 0
    total_length = 0
    for pred, ref in zip(predictions, references):
        total_distance += editdistance.eval(pred, ref)
        total_length += len(ref)
    return total_distance / max(total_length, 1)


def wer(predictions, references):
    """Word Error Rate global."""
    total_distance = 0
    total_length = 0
    for pred, ref in zip(predictions, references):
        pred_words = pred.split()
        ref_words = ref.split()
        total_distance += editdistance.eval(pred_words, ref_words)
        total_length += len(ref_words)
    return total_distance / max(total_length, 1)


cer_baseline = cer(predictions, references)
wer_baseline = wer(predictions, references)

print(f"═══════════════════════════════════════════════")
print(f"BASELINE TrOCR-base-handwritten — zéro fine-tuning")
print(f"═══════════════════════════════════════════════")
print(f"CER : {cer_baseline:.4f}  ({cer_baseline * 100:.2f} %)")
print(f"WER : {wer_baseline:.4f}  ({wer_baseline * 100:.2f} %)")
print(f"Sur {sample_size} lignes du split val")
print(f"═══════════════════════════════════════════════")
print()
print("Quelques exemples d'erreurs (référence vs prédiction) :")
print()
for i in range(min(5, sample_size)):
    print(f"  REF : {references[i][:80]}")
    print(f"  PRED: {predictions[i][:80]}")
    print()

## 10. Logger dans le journal d'expériences

Chaque run doit être enregistré dans `experiments/journal.jsonl`. C'est obligatoire pour la reproductibilité.

In [ ]:
# Journal d'expériences au format JSON Lines
run_entry = {
    "run_id": f"baseline_zero_shot_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    "timestamp": datetime.now().isoformat(),
    "model": "microsoft/trocr-base-handwritten",
    "fine_tuned": False,
    "sub_corpus": {
        "languages": TARGET_LANGUAGES,
        "centuries": TARGET_CENTURIES,
        "train_size": len(train_indices) if split_col else None,
        "val_size": len(val_indices) if split_col else None,
        "test_size": len(test_indices) if split_col else None,
        "test_set_sha256": test_hash,
    },
    "evaluation": {
        "split": "val",
        "sample_size": sample_size,
        "cer": cer_baseline,
        "wer": wer_baseline,
    },
    "seed": SEED,
    "device": DEVICE,
    "duration_seconds": elapsed,
    "notes": "Premier baseline zero-shot, avant tout fine-tuning",
}

print("Entrée à logger :")
print(json.dumps(run_entry, indent=2, ensure_ascii=False))
print()
print("📋 À COPIER dans experiments/journal.jsonl (une ligne JSON compacte par run)")
print()
# Version compacte une-ligne pour le JSONL
print("Version JSONL :")
print(json.dumps(run_entry, ensure_ascii=False))

## 11. Prochaines étapes

À partir d'ici :

1. **Coller le hash SHA-256** dans `DATA_SOURCES.md` et `MODEL_CARD.md`
2. **Coller l'entrée JSONL** dans `experiments/journal.jsonl`
3. **Implémenter le fine-tuning LoRA** dans `src/htr/finetune_trocr.py` :
   - Charger TrOCR
   - Wrapper avec `peft.LoraConfig(r=8, target_modules=...)`
   - Boucle d'entraînement avec early stopping sur val CER
   - Sauvegarder le checkpoint
4. **Évaluer le modèle fine-tuné** sur le val (pas le test !)
5. **Itérer** : ablations, augmentations, comparaison r=8 vs r=16
6. **Une seule fois, à la fin** : évaluer sur le test scellé

### Note sur les segments

CATMuS Medieval fournit les images au **niveau ligne**. Pour produire des polygones par ligne (exigence du brief pour l'IoU > 0,75), il faudra travailler sur les pages d'origine, accessibles via `CATMuS/medieval-segmentation` ou via les liens IIIF des manuscrits sources.

### Note sur le pipeline complet

Ce notebook ne couvre que la baseline HTR. Le pipeline complet exigé par le brief comprend en plus :
- Prétraitement (CLAHE, binarisation Sauvola)
- Segmentation layout
- Segmentation lignes (Kraken BLLA)
- Agrégation (si plusieurs modèles)
- Export PAGE XML / JSON

Ces composants vont dans les modules `src/preprocessing/`, `src/segmentation/`, `src/htr/`, `src/evaluation/`.

---

**Bon courage pour la suite. Pipeline avant perfection.**